In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-04-tanimoto-similarity.py"

# 兩個分子共用多少個指紋位元？
先用少數結構追蹤位元，再載入 ESOL 檢查同一個問題。首次執行會下載 ESOL。如要增寫練習程式，可接在分析儲存格的末尾，再重跑該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile
import deepchem as dc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, MACCSkeys, rdFingerprintGenerator

np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# 概念示意 — 實際應用請使用下方 DeepChem 版本
examples = {"Ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
            "Naproxen": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
            "2-Phenylpropanoic acid": "CC(C(=O)O)c1ccccc1"}
mols = [Chem.MolFromSmiles(smi) for smi in examples.values()]
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024, includeChirality=False)
fps = [gen.GetFingerprint(mol) for mol in mols]
a, b = (set(fp.GetOnBits()) for fp in fps[:2])
print(len(a), len(b), len(a & b), len(a | b))
print(DataStructs.TanimotoSimilarity(fps[0], fps[1]))
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3), layout="constrained")
labels = ["IBU", "NAP", "2-PPA"]
for ax, (name, pair_fps) in zip(axes, [("Morgan, radius 2", fps),
                                       ("MACCS", [MACCSkeys.GenMACCSKeys(m) for m in mols])]):
    matrix = np.array([DataStructs.BulkTanimotoSimilarity(fp, pair_fps) for fp in pair_fps])
    print(name, matrix.round(4))
    ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(3), labels)
    ax.set_yticks(range(3), labels)
    ax.set_title(name)
    for (i, j), score in np.ndenumerate(matrix):
        ax.text(j, i, f"{score:.3f}", ha="center", va="center",
                color="white" if score > 0.65 else "#142f45")
fig.savefig(figure_dir / "06-04-tanimoto-similarity.png", dpi=160)

# ESOL：1,128 筆實測水溶解度 log10(mol/L)，原始研究 DOI 10.1021/ci034243x。
# 只描述輸入結構，不訓練模型或轉換標籤。
tasks, splits, _ = dc.molnet.load_delaney(featurizer=dc.feat.RawFeaturizer(smiles=True),
    splitter="scaffold", transformers=[], data_dir=str(cache), reload=False)
esol_mols = [Chem.MolFromSmiles(smi) for split in splits for smi in split.ids]
assert len(esol_mols) == 1128 and all(mol is not None for mol in esol_mols)
ref = Chem.MolFromSmiles("S1C=NC2=C1C=CC=C2")
for name, make_fp in [("Morgan", gen.GetFingerprint), ("MACCS", MACCSkeys.GenMACCSKeys)]:
    esol_fps = [make_fp(mol) for mol in esol_mols]
    scores = np.array(DataStructs.BulkTanimotoSimilarity(make_fp(ref), esol_fps))
    print(name, "median", np.median(scores), "identical bit vectors", sum(scores == 1))

In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-04-tanimoto-similarity.png")))